# AIDev v3 — G0 historical cohort check (China Task Innovation Dynamics project)

**操作：点击 Runtime → Run all（运行所有单元格）。** 仅需免费 CPU 环境，不用上传、下载或修改文件。结果直接显示在最后一个单元格；把最后的完整输出复制回 ChatGPT。

Data: official [AIDev v3](https://huggingface.co/datasets/hao-li/AIDev/tree/v3); benchmark: Hindi et al. (2026) [IEEE Access](https://doi.org/10.1109/ACCESS.2026.3696573).

Scope: only `pull_request.parquet` (~16.9MB) and `repository.parquet` (~0.17MB), same v3 commit, repo stars ≥500, five predefined agent labels, PR created before Aug 2, 2025 (UTC). Published comparison counts: 12,433 PRs, 1,495 repos, 11,100 closed, 4,283 rejected. **These are not filters or research results.** No regression, model, or raw PR data export. A mismatch is an important finding requiring provenance audit, not a reason to change research cutoffs.

Current-public metadata/source vintage versus historical paper's as-of date, outcomes, repo popularity, and source agent attribution must be kept distinct. This script performs a low-cost G0 source feasibility check, **not a G0 PASS**.

In [ ]:
%pip -q install "huggingface_hub>=0.29" "pandas>=2" "pyarrow>=15"

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download

DS, TAG = "hao-li/AIDev", "v3"
FILENAMES = ("pull_request.parquet", "repository.parquet")
AGENTS = ("OpenAI_Codex", "Devin", "Copilot", "Cursor", "Claude_Code")
STAR_MIN = 500
END = pd.Timestamp("2025-08-02T00:00:00Z")
MODEL_START = pd.Timestamp("2024-12-01T00:00:00Z")
MODEL_END = pd.Timestamp("2025-08-01T00:00:00Z")
PUBLISHED = {"prs":12433, "repos":1495, "closed":11100, "rejected":4283}
LIMIT = 60 * 1024 * 1024

# Resolve the published historical tag to an immutable revision.
info = HfApi().repo_info(repo_id=DS, repo_type="dataset", revision=TAG, files_metadata=True)
revision = info.sha
if not revision or len(revision) != 40:
    raise RuntimeError("STOP: Cannot verify immutable v3 revision.")
inventory = {obj.rfilename: obj for obj in info.siblings or []}
for name in FILENAMES:
    if name not in inventory:
        raise RuntimeError(f"STOP: Missing {name} in v3. Do not use latest/main!")
    nbytes = getattr(inventory[name], "size", None)
    if nbytes is not None and nbytes > LIMIT:
        raise RuntimeError(f"STOP: {name} unexpectedly large: {nbytes} bytes")
print("Pinned official dataset:", DS, "tag:", TAG, "commit:", revision)

paths, metadata = {}, {}
for name in FILENAMES:
    file = Path(hf_hub_download(repo_id=DS, repo_type="dataset", filename=name, revision=revision))
    if file.stat().st_size > LIMIT:
        raise RuntimeError(f"STOP: downloaded file {name} exceeded size limit")
    digest = hashlib.sha256()
    with file.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    parquet = pq.ParquetFile(file)
    metadata[name] = {
        "bytes":file.stat().st_size, "sha256":digest.hexdigest(),
        "rows_footer":parquet.metadata.num_rows,
        "columns":parquet.schema.names,
    }
    paths[name] = file
print("\nSOURCE INTEGRITY (original bytes):")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

prs = pd.read_parquet(paths["pull_request.parquet"])
repos = pd.read_parquet(paths["repository.parquet"])
need_pr = {"id","repo_id","agent","created_at","closed_at","merged_at","state"}
need_repo = {"id","stars"}
missing_pr = sorted(need_pr - set(prs.columns))
missing_repo = sorted(need_repo - set(repos.columns))
if missing_pr or missing_repo:
    raise RuntimeError(f"STOP source schema mismatch. PR missing={missing_pr}; repo missing={missing_repo}")
for tab, key in ((prs,"id"),(repos,"id")):
    if tab[key].isna().any() or tab[key].duplicated().any():
        raise RuntimeError(f"STOP: missing/duplicated {key}; no automatic deduplication")

stars = pd.to_numeric(repos["stars"], errors="coerce")
pop_ids = set(repos.loc[stars >= STAR_MIN, "id"])
p = prs.loc[prs["repo_id"].isin(pop_ids) & prs["agent"].astype(str).isin(AGENTS)].copy()
created = pd.to_datetime(p["created_at"], utc=True, errors="coerce")
p = p.loc[created.lt(END)].copy()
created = pd.to_datetime(p["created_at"], utc=True, errors="coerce")
closed = pd.to_datetime(p["closed_at"], utc=True, errors="coerce")
merged = pd.to_datetime(p["merged_at"], utc=True, errors="coerce")
state_closed = p["state"].astype(str).str.lower().eq("closed")
closed_snapshot = state_closed | merged.notna()
rejected_snapshot = closed_snapshot & merged.isna()
closed_asof = closed.lt(END) | merged.lt(END)
merged_asof = merged.lt(END)
rejected_asof = closed_asof & ~merged_asof
in_model_months = created.ge(MODEL_START) & created.lt(MODEL_END)

observed = {
  "prs":int(len(p)),
  "repos":int(p["repo_id"].nunique()),
  "closed_snapshot":int(closed_snapshot.sum()),
  "rejected_snapshot":int(rejected_snapshot.sum()),
  "closed_asof":int(closed_asof.sum()),
  "rejected_asof":int(rejected_asof.sum()),
  "model_month_prs_dec2024_jul2025":int(in_model_months.sum()),
  "model_month_closed_snapshot":int((in_model_months & closed_snapshot).sum()),
  "model_month_closed_asof":int((in_model_months & closed_asof).sum()),
  "raw_pop_pr_rows":int(len(prs)),
  "raw_pop_repo_rows":int(len(repos)),
  "missing_repo_fk":int((~prs["repo_id"].isin(repos["id"])).sum()),
  "missing_stars":int(stars.isna().sum()),
  "missing_created_cohort":int(created.isna().sum()),
  "later_closures_in_snapshot":int((closed_snapshot & ~closed_asof).sum()),
  "agent_counts":{str(k):int(v) for k,v in p["agent"].value_counts().items()},
}

print("\n=== OBSERVED SOURCE COUNTS (not paper replication proof) ===")
print(json.dumps({"observed":observed,"published_comparison_not_filters":PUBLISHED},
                 indent=2, ensure_ascii=False))
print("\nCOMPARISON: total PR difference =",observed["prs"]-PUBLISHED["prs"],
      "; repo difference =",observed["repos"]-PUBLISHED["repos"])
print("RESEARCH STOP RULE: matching counts alone does not verify original PR IDs,")
print("historical star snapshot, agent attribution, outcome as-of timing, or regression.")
print("G0 source check only. No fitted model, no G0/G1 PASS.")
